# Diabetes screening model review\n\nThis notebook is a reproducible training/evaluation template. It does not fabricate data or metrics.\n

In [ ]:
from __future__ import annotations

import os
from pathlib import Path

import numpy as np
import pandas as pd
from diabetes_contracts import FEATURES
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler


In [ ]:
TARGET = 'Outcome'
RANDOM_STATE = 42
DATA_PATH = Path(os.environ.get('DIABETES_DATA_PATH', 'data/diabetes.csv')).expanduser()
if not DATA_PATH.is_absolute():
    DATA_PATH = Path.cwd() / DATA_PATH
if not DATA_PATH.is_file():
    raise FileNotFoundError(
        f'Dataset not found at {DATA_PATH}. Set DIABETES_DATA_PATH to the CSV path.'
    )
df = pd.read_csv(DATA_PATH)
required = FEATURES + [TARGET]
if df.columns.duplicated().any():
    raise ValueError('Dataset contains duplicate column names.')
missing = set(required) - set(df.columns)
if missing:
    raise ValueError(f'Dataset is missing required columns: {sorted(missing)}')
df = df[required].copy()
for column in FEATURES:
    converted = pd.to_numeric(df[column], errors='coerce')
    invalid = df[column].notna() & converted.isna()
    if invalid.any():
        raise ValueError(f'{column} contains non-numeric values.')
    df[column] = converted
feature_values = df[FEATURES].to_numpy(dtype=float, na_value=np.nan)
if not np.isfinite(feature_values[~np.isnan(feature_values)]).all():
    raise ValueError('Feature values must be finite when present; missing values are allowed for imputation.')
target = pd.to_numeric(df[TARGET], errors='coerce')
if target.isna().any() or not target.isin([0, 1]).all():
    raise ValueError('Outcome must be non-null and contain only binary values 0 and 1.')
df[TARGET] = target.astype('int64')
feature_label_counts = df.groupby(FEATURES, dropna=False, sort=False)[TARGET].nunique()
if (feature_label_counts > 1).any():
    raise ValueError('Identical feature rows have conflicting Outcome labels.')
df = df.drop_duplicates(subset=FEATURES).copy()
if any(df[column].notna().sum() == 0 for column in FEATURES):
    raise ValueError('Every feature must contain at least one observed value.')
if len(df) < 10:
    raise ValueError('Dataset must contain at least 10 unique rows for a held-out split.')


In [ ]:
X = df[FEATURES]
y = df[TARGET]
if y.nunique() != 2 or y.value_counts().min() < 2:
    raise ValueError('Outcome must contain both classes with at least two rows per class.')
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=RANDOM_STATE
)


In [ ]:
# Fit all learned preprocessing only on the training partition.
model = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
    ('classifier', LogisticRegression(max_iter=2000, class_weight='balanced', random_state=RANDOM_STATE)),
])
model.fit(X_train, y_train)
predicted = model.predict(X_test)
probability = model.predict_proba(X_test)[:, 1]


In [ ]:
metrics = {
    'accuracy': accuracy_score(y_test, predicted),
    'recall': recall_score(y_test, predicted),
    'f1': f1_score(y_test, predicted),
    'roc_auc': roc_auc_score(y_test, probability),
}
print(metrics)
print(confusion_matrix(y_test, predicted))
print(classification_report(y_test, predicted, zero_division=0))


## Reproducibility notes\n\nThe test split is stratified and seeded. Metrics are calculated with `y_test` as the ground truth. No oversampling is applied; if it is added later, it must be fitted inside the training pipeline only. The checked-in `model.pkl` is an existing inference artifact and is not overwritten by this notebook.\n